DEFINE AND VISUALISE REGIONS OF INTEREST

In [ ]:
import mne
from mne.viz import Brain
import os
import numpy as np
import pickle

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config import check_paths
from config.config import HEMI, ROI, SUBJECTS, FS_SUB, FS_SRC_PATH, GAMMA, COUPLINGS
from config.paths import FS_FOLDER, SOURCE_DATA_DIR, ROI_STCS_DIR

import os
import numpy as np
import matplotlib.pyplot as plt
from tensorpac import EventRelatedPac

%matplotlib qt

In [ ]:
labels = mne.read_labels_from_annot(
    subject=FS_SUB,
    parc="aparc.a2009s", # "aparc.a2009s", "Yeo2011_7Networks_N1000", "Yeo2011_17Networks_N1000"
    hemi=HEMI[1:],
    subjects_dir=FS_FOLDER
)

label_dict = {label.name: label for label in labels}

ROI_labels = {
    roi: {
        label_name: label_dict[label_name]
        for label_name in label_names
    }
    for roi, label_names in ROI.items()
}

for label in labels:
    print(label.name)

# # OPTIONAL: plot label borders on the brain surface
# colors = {
#     "M1": "red",
#     "S1": "blue",
#     "PMC": "green",
#     "SMA": "purple",
# }

# brain = Brain(
#     FS_SUB,
#     hemi=HEMI[1:],
#     surf="pial",
#     subjects_dir=FS_FOLDER,
#     background="black"
# )

# for roi, label_names in ROI.items():
#     for label_name in label_names:
#         brain.add_label(label_dict[label_name], borders=False, color=colors[roi])


EXTRACT SOURCE TIME SERIES BY LABEL

In [ ]:
src = mne.read_source_spaces(FS_SRC_PATH)


    Reading a source space...
    Computing patch statistics...
    Patch information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    [done]
    2 source spaces read


In [ ]:
source_path = "F:\\# study 2\\eeg_data\\epochs\\source"
group = "Y"
sub = "s1_pac_sub24"
task = "FTT"
stage = "plan"
source_path = os.path.join(SOURCE_DATA_DIR, group, sub, task, stage)

epo_stcs = [mne.read_source_estimate(os.path.join(source_path, fname_stc)) for fname_stc in os.listdir(source_path) if fname_stc.endswith('-lh.stc')]
epo_stcs[0]

In [134]:
ROI_labels

{'M1': {'G_precentral-lh': <Label | fsaverage_bem, 'G_precentral-lh', lh : 3876 vertices>},
 'S1': {'G_postcentral-lh': <Label | fsaverage_bem, 'G_postcentral-lh', lh : 3509 vertices>,
  'S_postcentral-lh': <Label | fsaverage_bem, 'S_postcentral-lh', lh : 4447 vertices>},
 'PMC': {'S_precentral-sup-part-lh': <Label | fsaverage_bem, 'S_precentral-sup-part-lh', lh : 1846 vertices>,
  'S_precentral-inf-part-lh': <Label | fsaverage_bem, 'S_precentral-inf-part-lh', lh : 1881 vertices>,
  'G_front_inf-Opercular-lh': <Label | fsaverage_bem, 'G_front_inf-Opercular-lh', lh : 1800 vertices>},
 'SMA': {'G_and_S_paracentral-lh': <Label | fsaverage_bem, 'G_and_S_paracentral-lh', lh : 2272 vertices>,
  'G_front_sup-lh': <Label | fsaverage_bem, 'G_front_sup-lh', lh : 8394 vertices>}}

In [192]:
for roi, labels in ROI_labels.items():

    print(f"Processing ROI: {roi}")

    for label_name, label in labels.items():
        print(f"  Processing label: {label_name}")

Processing ROI: M1
  Processing label: G_precentral-lh
Processing ROI: S1
  Processing label: G_postcentral-lh
  Processing label: S_postcentral-lh
Processing ROI: PMC
  Processing label: S_precentral-sup-part-lh
  Processing label: S_precentral-inf-part-lh
  Processing label: G_front_inf-Opercular-lh
Processing ROI: SMA
  Processing label: G_and_S_paracentral-lh
  Processing label: G_front_sup-lh


In [ ]:
roi_data = {}

for roi, labels in ROI_labels.items():

    roi_data[roi] = {}

    for label_name, label in labels.items():

        stcs_label = [stc.in_label(label) for stc in epo_stcs]

        roi_data[roi][label_name] = {
            "data": np.stack([stc.data for stc in stcs_label]),
            "vertices": stcs_label[0].vertices[0],
            "hemisphere": HEMI[1:],
            "times": stcs_label[0].times,
            "tstep": stcs_label[0].tstep,
            "sfreq": stcs_label[0].sfreq,
        }

roi_data

{'M1': {'G_precentral-lh': {'data': array([[[-5.89796007e-02, -3.00747007e-02,  3.81895974e-02, ...,
             3.06706190e-01,  1.86369091e-01,  9.67492461e-02],
           [-1.48766249e-01, -2.66776472e-01, -2.27447122e-01, ...,
            -1.67692900e-01,  2.05570832e-01,  4.35404718e-01],
           [-5.16671538e-02, -4.08505760e-02, -4.91171740e-02, ...,
            -1.46756083e-01, -1.47479475e-01, -1.12603329e-01],
           ...,
           [-4.74931411e-02, -3.80117167e-03,  3.27860229e-02, ...,
             2.73040563e-01,  2.07324013e-01,  1.65326655e-01],
           [ 3.17656666e-01,  2.86941260e-01,  1.35788128e-01, ...,
            -5.77007011e-02, -2.94542294e-02, -5.50838895e-02],
           [ 8.05290937e-02,  7.66360164e-02,  1.87161155e-02, ...,
            -4.71026786e-02,  5.70299067e-02,  1.84942141e-01]],
   
          [[ 1.83201700e-01,  1.68715015e-01,  1.19594626e-01, ...,
            -2.65904903e-01, -3.12137097e-01, -3.19755942e-01],
           [-2.3669916

In [ ]:
roi_save_dir = os.path.join(ROI_STCS_DIR, group, task, stage)
check_paths(roi_save_dir)

with open(os.path.join(roi_save_dir, f"{sub}_{task}_{stage}_roi_stcs.pkl"), "wb") as f:
    pickle.dump(roi_data, f)


VIZ

In [189]:
roi_data

{'M1': {'G_precentral-lh': {'data': array([[[-5.89796007e-02, -3.00747007e-02,  3.81895974e-02, ...,
             3.06706190e-01,  1.86369091e-01,  9.67492461e-02],
           [-1.48766249e-01, -2.66776472e-01, -2.27447122e-01, ...,
            -1.67692900e-01,  2.05570832e-01,  4.35404718e-01],
           [-5.16671538e-02, -4.08505760e-02, -4.91171740e-02, ...,
            -1.46756083e-01, -1.47479475e-01, -1.12603329e-01],
           ...,
           [-4.74931411e-02, -3.80117167e-03,  3.27860229e-02, ...,
             2.73040563e-01,  2.07324013e-01,  1.65326655e-01],
           [ 3.17656666e-01,  2.86941260e-01,  1.35788128e-01, ...,
            -5.77007011e-02, -2.94542294e-02, -5.50838895e-02],
           [ 8.05290937e-02,  7.66360164e-02,  1.87161155e-02, ...,
            -4.71026786e-02,  5.70299067e-02,  1.84942141e-01]],
   
          [[ 1.83201700e-01,  1.68715015e-01,  1.19594626e-01, ...,
            -2.65904903e-01, -3.12137097e-01, -3.19755942e-01],
           [-2.3669916

In [ ]:

for roi, labels in roi_data.items():

    for label_name, label_data in labels.items():

        data = label_data["data"]      # epochs x vertices x times
        times = label_data["times"]

        # average across epochs and vertices
        tc = data.mean(axis=(0, 1))

        fig, ax = plt.subplots(figsize=(8, 3))

        ax.plot(times, tc)
        ax.axvline(0, color="k", linestyle="--")
        ax.set_ylim(-0.03, 0.05)

        ax.set_title(f"{roi}: {label_name}")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Source amplitude")

        plt.show()

ERPAC

In [ ]:
# file check
roi_save_dir = os.path.join(ROI_STCS_DIR, group, task, stage)
with open(os.path.join(roi_save_dir, f"{sub}_{task}_{stage}_roi_stcs.pkl"), "rb") as f:
    roi_data = pickle.load(f)

roi_data

{'M1': {'G_precentral-lh': {'data': array([[[-5.89796007e-02, -3.00747007e-02,  3.81895974e-02, ...,
             3.06706190e-01,  1.86369091e-01,  9.67492461e-02],
           [-1.48766249e-01, -2.66776472e-01, -2.27447122e-01, ...,
            -1.67692900e-01,  2.05570832e-01,  4.35404718e-01],
           [-5.16671538e-02, -4.08505760e-02, -4.91171740e-02, ...,
            -1.46756083e-01, -1.47479475e-01, -1.12603329e-01],
           ...,
           [-4.74931411e-02, -3.80117167e-03,  3.27860229e-02, ...,
             2.73040563e-01,  2.07324013e-01,  1.65326655e-01],
           [ 3.17656666e-01,  2.86941260e-01,  1.35788128e-01, ...,
            -5.77007011e-02, -2.94542294e-02, -5.50838895e-02],
           [ 8.05290937e-02,  7.66360164e-02,  1.87161155e-02, ...,
            -4.71026786e-02,  5.70299067e-02,  1.84942141e-01]],
   
          [[ 1.83201700e-01,  1.68715015e-01,  1.19594626e-01, ...,
            -2.65904903e-01, -3.12137097e-01, -3.19755942e-01],
           [-2.3669916

In [ ]:
sf = roi_data["M1"]["G_precentral-lh"]["sfreq"] # extract from any ROI and label, since they all have the same sampling frequency
alpha_threshold = 0.05


In [ ]:
for coupling_name, (phase_freq, amp_freq) in COUPLINGS.items():
    print(f"Processing coupling: {coupling_name}")
    print(f"Phase frequency: {phase_freq}, Amplitude frequency: {amp_freq}")

Processing coupling: theta_gamma
Phase frequency: [4, 8], Amplitude frequency: (30, 80, 5, 1)
Processing coupling: alpha_gamma
Phase frequency: [8, 12], Amplitude frequency: (30, 80, 5, 1)
Processing coupling: beta_gamma
Phase frequency: [13, 30], Amplitude frequency: (30, 80, 5, 1)


In [20]:
for roi, labels in roi_data.items():
    print(f"Processing ROI: {roi}")
    for label, data in labels.items():
        print(f"  Processing label: {label}")

Processing ROI: M1
  Processing label: G_precentral-lh
Processing ROI: S1
  Processing label: G_postcentral-lh
  Processing label: S_postcentral-lh
Processing ROI: PMC
  Processing label: S_precentral-sup-part-lh
  Processing label: S_precentral-inf-part-lh
  Processing label: G_front_inf-Opercular-lh
Processing ROI: SMA
  Processing label: G_and_S_paracentral-lh
  Processing label: G_front_sup-lh


In [22]:
for label_name, label_data in labels.items():

            data = label_data["data"]
            print(f"Processing label: {label_name}, data shape: {data.shape}")

Processing label: G_and_S_paracentral-lh, data shape: (99, 35, 501)
Processing label: G_front_sup-lh, data shape: (99, 134, 501)


In [ ]:
erpac_results = {}

for coupling_name, (phase_freq, amp_freq) in COUPLINGS.items():

    print(f"\nComputing {coupling_name}")

    p = EventRelatedPac(
        f_pha=phase_freq,
        f_amp=amp_freq
    )
    print(f"Phase freq is {p.xvec}")

    erpac_results[coupling_name] = {}

    for roi, labels in roi_data.items():

        erpac_results[coupling_name][roi] = {}

        for label_name, label_data in labels.items():

            data = label_data["data"]      # (epochs, vertices, times)

            n_epochs, n_vertices, n_times = data.shape
            n_freqs = len(p.yvec)

            vertex_erpac = np.zeros((n_vertices, n_freqs, n_times))
            vertex_sig = np.zeros((n_vertices, n_times))

            for v in range(n_vertices):

                x = data[:, v, :]

                erpac = p.filterfit(
                    sf,
                    x,
                    method="circular",
                    mcp="bonferroni",
                    n_jobs=-1
                ).squeeze()

                pvalues = p.pvalues.squeeze()

                vertex_erpac[v] = erpac # (n_amp, n_times)

            erpac_results[coupling_name][roi][label_name] = {
                "erpac": vertex_erpac,
                "p-values": pvalues,
                "times": label_data["times"],
                "vertices": label_data["vertices"],
            }

In [ ]:
with open(os.path.join(roi_save_dir, f"{sub}_{task}_{stage}_erpac_results.pkl"), "wb") as f:
    pickle.dump(erpac_results, f)

In [44]:
erpac_results

{'theta_gamma': {'M1': {'G_precentral-lh': {'erpac': array([[[0.1359138 , 0.12852219, 0.08784582, ..., 0.21934076,
             0.15589338, 0.13214091],
            [0.1658135 , 0.09404459, 0.03966389, ..., 0.24100336,
             0.18343524, 0.09100494],
            [0.20441481, 0.03999144, 0.02847558, ..., 0.23187644,
             0.18779897, 0.03113341],
            ...,
            [0.10496315, 0.15899976, 0.18079246, ..., 0.05905107,
             0.05810336, 0.30233934],
            [0.09166635, 0.17185537, 0.19226228, ..., 0.07031122,
             0.06366808, 0.28889359],
            [0.08341355, 0.18607523, 0.20165358, ..., 0.08161762,
             0.0700153 , 0.26781874]],
    
           [[0.18625926, 0.13490317, 0.14358047, ..., 0.05200378,
             0.04794782, 0.22739389],
            [0.16626883, 0.1253137 , 0.12265985, ..., 0.07397392,
             0.08527494, 0.27024508],
            [0.1451428 , 0.14083569, 0.13704666, ..., 0.08940286,
             0.09851403, 0.274

In [ ]:
with open(os.path.join(roi_save_dir, f"{sub}_{task}_{stage}_erpac_results.pkl"), "rb") as f:
    erpac_results = pickle.load(f)

erpac_results

{'theta_gamma': {'M1': {'G_precentral-lh': {'erpac': array([[[0.1359138 , 0.12852219, 0.08784582, ..., 0.21934076,
             0.15589338, 0.13214091],
            [0.1658135 , 0.09404459, 0.03966389, ..., 0.24100336,
             0.18343524, 0.09100494],
            [0.20441481, 0.03999144, 0.02847558, ..., 0.23187644,
             0.18779897, 0.03113341],
            ...,
            [0.10496315, 0.15899976, 0.18079246, ..., 0.05905107,
             0.05810336, 0.30233934],
            [0.09166635, 0.17185537, 0.19226228, ..., 0.07031122,
             0.06366808, 0.28889359],
            [0.08341355, 0.18607523, 0.20165358, ..., 0.08161762,
             0.0700153 , 0.26781874]],
    
           [[0.18625926, 0.13490317, 0.14358047, ..., 0.05200378,
             0.04794782, 0.22739389],
            [0.16626883, 0.1253137 , 0.12265985, ..., 0.07397392,
             0.08527494, 0.27024508],
            [0.1451428 , 0.14083569, 0.13704666, ..., 0.08940286,
             0.09851403, 0.274

In [ ]:
# Compute mean PAC values for each ROI - !!! do prior to saving individual subs data to a dataset with all subjects !!!
roi_mean = {}

for roi, labels in erpac_results["theta_gamma"].items():

    label_means = []

    for label_name, res in labels.items():
        # average across vertices
        label_means.append(
            res["erpac"].mean(axis=0)     # (freqs, times)
        )

    # average anatomical labels
    roi_mean[roi] = np.mean(label_means, axis=0)

roi_mean["M1"].shape

(45, 501)

ERPAC VIZ

In [ ]:
# !!! use after group averaging !!!

for coupling in erpac_results:

    # Average ERPAC across vertices, then across anatomical labels
    roi_mean = {}

    for roi, labels in erpac_results[coupling].items():

        label_means = [
            res["erpac"].mean(axis=0)      # (gamma_freqs, times)
            for res in labels.values()
        ]

        roi_mean[roi] = np.mean(label_means, axis=0)

    # use times from any ROI
    first_roi = next(iter(erpac_results[coupling]))
    first_label = next(iter(erpac_results[coupling][first_roi]))
    times = erpac_results[coupling][first_roi][first_label]["times"]

    # Plot
    fig, axes = plt.subplots(
        2, 2,
        figsize=(10, 8),
        sharex=True,
        sharey=True,
        constrained_layout=True
    )

    for ax, roi in zip(axes.ravel(), roi_mean):

        im = ax.imshow(
            roi_mean[roi],
            aspect="auto",
            origin="lower",
            extent=[times[0], times[-1], GAMMA[0], GAMMA[1]]
        )

        ax.set_title(roi)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Gamma (Hz)")

    fig.suptitle(coupling.replace("_", " ").title())

    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="ERPAC")

    plt.show()

____

Plotting drafts

In [ ]:
import mne
from mne.viz import Brain
import os
import numpy as np
import pickle
import pyarrow

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config import check_paths
from config.config import (HEMI, ROI, FS_SUB, FS_SRC_PATH, COUPLINGS,
                           SUBJECTS, GROUPS, TASKS, TASK_STAGES)
from config.paths import FS_FOLDER, SOURCE_DATA_DIR, ERPAC_DIR, ROI_STCS_DIR, ERPAC_FIGS_DIR
from utils.helpers import iterate_dataset
from utils.plotting import plot_group_erpac

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorpac import EventRelatedPac

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

% matplotlib qt

# =========================
# SOURCE ROI PLOTTING CONFIG
# =========================

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# =========================
# SOURCE ROI PLOTTING CONFIG
# =========================

def plot_group_erpac(
    erpac_df,
    task_stage,
    coupling,
    group=None,
    save_dir=None,
    dpi=300
):

    min_value = 0.11
    max_value = 0.14
    step = 0.005

    # Filter data
    df = erpac_df[
        (erpac_df["task_stage"] == task_stage) &
        (erpac_df["coupling"] == coupling)
    ]

    if group is not None:
        df = df[df["group"] == group]


    rois = df["roi"].unique()


    fig, axes = plt.subplots(
        2, 2,
        figsize=(10, 8),
        sharex=True,
        sharey=True,
        constrained_layout=True
    )

    axes = axes.ravel()


    for ax, roi in zip(axes, rois):

        roi_df = df[df["roi"] == roi]


        # Average across subjects
        roi_mean = (
            roi_df
            .groupby(
                ["amp_freq", "time"]
            )["erpac_value"]
            .mean()
            .reset_index()
        )


        # Convert long format → matrix
        erpac_matrix = (
            roi_mean
            .pivot(
                index="amp_freq",
                columns="time",
                values="erpac_value"
            )
            .values
        )


        amp_freqs = roi_mean["amp_freq"].unique()
        times = roi_mean["time"].unique()


        im = ax.imshow(
            erpac_matrix,
            aspect="auto",
            origin="lower",
            extent=[
                times[0],
                times[-1],
                amp_freqs[0],
                amp_freqs[-1]
            ],
            vmin=min_value,
            vmax=max_value
        )


        ax.set_title(roi)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Gamma (Hz)")


    # Remove empty axes
    for ax in axes[len(rois):]:
        ax.remove()


    title = (
        f"{group + ' - ' if group else ''}"
        f"{task_stage} - "
        f"{coupling.replace('_', ' ').title()}"
    )

    fig.suptitle(title)

    fig.colorbar(
        im,
        ax=axes.tolist(),
        shrink=0.8,
        label="ERPAC",
        ticks=np.arange(min_value, max_value, step)
    )

    # Save figure
    if save_dir is not None:

        save_dir = Path(save_dir)
        save_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        group_name = group if group else "all"

        fname = (
            f"{group_name}_"
            f"{task_stage}_"
            f"{coupling}_"
            "ERPAC.png"
        )

        fig.savefig(
            save_dir / fname,
            dpi=dpi,
            bbox_inches="tight"
        )

    plt.show()
    plt.close()

erpac_df = pd.read_parquet(
    os.path.join(
        ERPAC_DIR,
        "erpac_results.parquet"
    ),
    engine="pyarrow"
)

for group in GROUPS:
    for task_stage in TASK_STAGES:
        for coupling in COUPLINGS:
            plot_group_erpac(
                erpac_df,
                task_stage=task_stage,
                coupling=coupling,
                group=group,
                save_dir=ERPAC_FIGS_DIR
            )

In [40]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np


def plot_group_erpac_timecourse(
    erpac_df,
    task_stage,
    coupling,
    mode="mean",
    save_dir=None,
    dpi=300,
):
    """
    Plot ROI ERPAC timecourses for Young and Older groups.

    Parameters
    ----------
    erpac_df : pandas.DataFrame
        Long-format ERPAC dataframe.

    task_stage : str
        Experimental stage (e.g. "plan", "go").

    coupling : str
        Coupling name (e.g. "theta_gamma").

    mode : {"mean", "max"}
        mean      -> mean ERPAC across gamma frequencies
        max       -> maximum ERPAC across gamma frequencies

    save_dir : str or Path | None
        Directory for saving figure.

    dpi : int
        Figure resolution.
    """

    # -------------------------------------------------------
    # Filter dataframe
    # -------------------------------------------------------

    df = erpac_df[
        (erpac_df["task_stage"] == task_stage) &
        (erpac_df["coupling"] == coupling)
    ]

    rois = sorted(df["roi"].unique())
    groups = sorted(df["group"].unique())

    y_min = 0.115
    y_max = 0.15

    # -------------------------------------------------------
    # Create figure
    # -------------------------------------------------------

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(10, 8),
        sharex=True,
        sharey=True,
        constrained_layout=True
    )

    axes = axes.ravel()

    # -------------------------------------------------------
    # Plot each ROI
    # -------------------------------------------------------

    for ax, roi in zip(axes, rois):

        for group in groups:

            roi_df = df[
                (df["roi"] == roi) &
                (df["group"] == group)
            ]

            # ---------------------------------------------
            # Group-average ERPAC surface
            # ---------------------------------------------

            roi_mean = (
                roi_df
                .groupby(
                    ["amp_freq", "time"]
                )["erpac_value"]
                .mean()
                .reset_index()
            )

            erpac_matrix = (
                roi_mean
                .pivot(
                    index="amp_freq",
                    columns="time",
                    values="erpac_value"
                )
                .sort_index()
            )

            times = erpac_matrix.columns.values

            matrix = erpac_matrix.values

            # ---------------------------------------------
            # Collapse frequency dimension
            # ---------------------------------------------

            if mode == "max":

                y = matrix.max(axis=0)
                ylabel = "Maximum ERPAC"

            elif mode == "mean":

                y = matrix.mean(axis=0)
                ylabel = "Mean ERPAC"

            else:

                raise ValueError(
                    "mode must be 'max' or 'mean'"
                )

            # ---------------------------------------------
            # Plot
            # ---------------------------------------------

            ax.plot(
                times,
                y,
                linewidth=2,
                label=group
            )

        ax.set_title(roi)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel(ylabel)

        ax.axvline(
            0,
            color="k",
            linestyle="--",
            linewidth=1
        )

        ax.set_xlim(times[0], times[-1])
        ax.set_ylim(y_min, y_max)

        ax.legend()

    # Remove unused axes

    for ax in axes[len(rois):]:
        ax.remove()

    fig.suptitle(
        f"{task_stage} | "
        f"{coupling.replace('_', ' ').title()} | "
        f"{mode.replace('_', ' ').title()}"
    )

    # -------------------------------------------------------
    # Save
    # -------------------------------------------------------

    if save_dir is not None:

        save_dir = Path(save_dir)
        save_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        fname = (
            f"{task_stage}_"
            f"{coupling}_"
            f"{mode}_"
            "timecourse.png"
        )

        fig.savefig(
            save_dir / fname,
            dpi=dpi,
            bbox_inches="tight"
        )

    plt.show()
    plt.close()

In [41]:
for task_stage in TASK_STAGES:
    for coupling in COUPLINGS:
        plot_group_erpac_timecourse(
            erpac_df,
            task_stage=task_stage,
            coupling=coupling,
            save_dir=ERPAC_FIGS_DIR,
            mode="max" 
        )